# Process Window — Dose-Focus Matrix

This tutorial computes a **Bossung plot** (CD vs dose × focus) and extracts
**Depth of Focus (DoF)** and **Exposure Latitude (EL)**.

The process window defines the range of dose and focus that keeps CD within
±10 % of the target. DoF is the in-spec focus range at the best dose; EL is the
in-spec dose range at best focus divided by the best dose, in %. Both come from
`euvsimulator.metro.process_window`, the same definition `euv process-window` uses.

The sweep runs 10 × 7 = 70 simulations, well under a minute on CPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from euvsimulator.metro.process_window import process_window
from euvsimulator.pipeline import SimulationConfig, run_simulation

In [ ]:
# Dose-focus grid. The dose-to-size of this pattern is about 13 mJ/cm², so the doses
# step by 1 mJ/cm² around it: with 5 mJ/cm² steps from 10 mJ/cm² only one dose prints
# in spec and EL is 0 by definition. Both ranges reach beyond the in-spec window,
# otherwise EL or DoF would just be the scan range.
doses = np.linspace(10, 19, 10)      # 10 to 19 mJ/cm²
focuses = np.linspace(-150, 150, 7)  # -150 to +150 nm defocus
target_cd = 32.0

cd_matrix = np.zeros((len(doses), len(focuses)))

for i, d in enumerate(doses):
    for j, f in enumerate(focuses):
        cfg = SimulationConfig(
            period_nm=64.0,
            line_width_nm=target_cd,
            dose_mj_cm2=d,
            focus_nm=f,
            na=0.33,
            grid=256,
        )
        result = run_simulation(cfg)
        cd_matrix[i, j] = result.cd_nm
        print(f"d={d:.0f}, f={f:+4.0f} → CD={result.cd_nm:.2f}")

In [ ]:
# Bossung plot
plt.figure(figsize=(10, 6))
for i, d in enumerate(doses):
    plt.plot(focuses, cd_matrix[i], 'o-', label=f'{d:.0f} mJ/cm²')

plt.axhline(target_cd * 0.9, color='gray', ls='--', alpha=0.5, label='±10 % spec')
plt.axhline(target_cd * 1.1, color='gray', ls='--', alpha=0.5)
plt.xlabel('Defocus [nm]')
plt.ylabel('CD [nm]')
plt.title('Bossung Plot — CD vs Focus at Various Doses')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Process window metrics. EL divides the in-spec dose range by the best dose (until
# 2026-10 this tutorial divided it by the target CD in nm, a unit error).
# process_window() expects the CD matrix as (focus, dose).
pw = process_window(cd_matrix.T, list(doses), list(focuses), target_cd=target_cd, tolerance=0.1)
dof = pw["dof_nm"]
el = pw["el_pct"]

print(f"Target CD:         {target_cd:.0f} nm (±10 %)")
print(f"Best focus:        {pw['best_focus']:+.0f} nm")
print(f"Best dose:         {pw['best_dose']:.1f} mJ/cm²")
print(f"Depth of Focus:    {dof:.1f} nm")
print(f"Exposure Latitude: {el:.1f} %")